# Lab 7 - Sequential orchestration

## What will you do?

Lab 6 chained two agents by hand. Here you use Agent Framework's dedicated **`SequentialBuilder`** to run a three-agent pipeline: draft an answer, simplify it, then safety-check it.

![Diagram that shows sequential orchestration where agents process tasks in a defined pipeline order. Output flows from one agent to the next.](https://learn.microsoft.com/en-us/azure/architecture/ai-ml/guide/_images/sequential-pattern.svg)

*Sequential orchestration. Source: [AI Agent Orchestration Patterns](https://learn.microsoft.com/en-us/azure/architecture/ai-ml/guide/ai-agent-design-patterns#sequential-orchestration) on Microsoft Learn.*

```text
question  ->  drafter  ->  simplifier  ->  safety reviewer  ->  answer
```

Each agent only sees the previous agent's output. There is no knowledge base tool in this lab; the point is the **pipeline**, not retrieval.

> **This is a workshop exercise, not a clinical tool.**

## Before you start

- Python 3.11 or later, with a notebook kernel selected, and `az login` completed.
- Lab 1 finished: a project endpoint and an approved model deployment.
- No knowledge base or tools are needed for this lab.

Install the pinned package set below. If you already imported a different version of these SDKs in this kernel, restart the kernel after installing.

In [ ]:
%pip install -q "azure-ai-projects==2.3.0" "azure-identity==1.25.3" "openai==2.54.0" "agent-framework-core==1.16.0" "agent-framework-openai==1.14.1" "agent-framework-foundry==1.11.0" "agent-framework-orchestrations==1.1.1"

## 0. Connect to your project

Use your Lab 1 settings. The cell signs in with your Azure CLI account and generates a suffix to keep your agents unique in the shared project.

In [ ]:
import asyncio
import os
import sys
from contextlib import AsyncExitStack
from uuid import uuid4

from agent_framework.foundry import FoundryAgent
from agent_framework.orchestrations import SequentialBuilder
from azure.ai.projects import AIProjectClient
from azure.ai.projects.models import PromptAgentDefinition
from azure.identity import AzureCliCredential

# Your nonsecret settings from Lab 1. Paste them between the quotes, or set them as
# environment variables before starting the kernel.
PROJECT_ENDPOINT = os.getenv("AZURE_AI_PROJECT_ENDPOINT", "")
MODEL_DEPLOYMENT = os.getenv("AZURE_AI_MODEL_DEPLOYMENT_NAME", "")

if sys.version_info < (3, 11):
    raise RuntimeError(
        "These notebooks need Python 3.11 or later. This kernel is "
        f"{sys.version_info.major}.{sys.version_info.minor}. Select a newer kernel."
    )

missing = [
    name
    for name, value in {
        "AZURE_AI_PROJECT_ENDPOINT": PROJECT_ENDPOINT,
        "AZURE_AI_MODEL_DEPLOYMENT_NAME": MODEL_DEPLOYMENT,
    }.items()
    if not value
]
if missing:
    raise ValueError(f"Set these before continuing: {', '.join(missing)}")


def check_todos(**answers: object) -> None:
    """Helper. Stops the cell while a `...` blank is still open."""
    still_open = [name for name, value in answers.items() if value is ...]
    if still_open:
        raise ValueError(f"Fill in these blanks first: {', '.join(still_open)}")


SUFFIX = uuid4().hex[:8]
credential = AzureCliCredential()
project = AIProjectClient(endpoint=PROJECT_ENDPOINT, credential=credential)
print(f"Setup complete. Suffix: {SUFFIX}")

## 1. Register three agents, one per pipeline stage

| | Drafter | Simplifier | Safety reviewer |
|---|---|---|---|
| Input | The patient question | The draft answer | The simplified answer |
| Produces | A clinically accurate draft | The same content in plain language | A finalized answer, with a safety disclaimer added if missing |

### To-Do 1 - Order the pipeline

**Goal:** three agents registered, ready to connect in the order the pipeline requires.

**Steps**

1. Set `PIPELINE_ORDER` to the three agent-name variables, **in the order the pipeline should run**.
2. Run the cell.

**Run the cell. You should see** three agent names printed in `PIPELINE_ORDER` order.

<details><summary>Hint</summary>

A draft must exist before it can be simplified, and a safety check needs the simplified text, not the raw draft.

</details>

<details><summary>Show solution code</summary>

```python
PIPELINE_ORDER = [drafter_version.name, simplifier_version.name, reviewer_version.name]
```

</details>

In [ ]:
DRAFTER_INSTRUCTIONS = (
    "You draft a clinically accurate answer to a patient's question, using plain clinical "
    "knowledge. Keep it under 120 words."
)
SIMPLIFIER_INSTRUCTIONS = (
    "You rewrite the draft you receive in plain language a patient without a medical background "
    "can understand. Do not add or remove medical claims."
)
REVIEWER_INSTRUCTIONS = (
    "You review the simplified answer you receive. If it is missing a line telling the reader to "
    "consult their own clinician, add one. Otherwise return the text unchanged."
)

drafter_version = project.agents.create_version(
    agent_name=f"day2-answer-drafter-{SUFFIX}",
    definition=PromptAgentDefinition(model=MODEL_DEPLOYMENT, instructions=DRAFTER_INSTRUCTIONS),
)
simplifier_version = project.agents.create_version(
    agent_name=f"day2-answer-simplifier-{SUFFIX}",
    definition=PromptAgentDefinition(model=MODEL_DEPLOYMENT, instructions=SIMPLIFIER_INSTRUCTIONS),
)
reviewer_version = project.agents.create_version(
    agent_name=f"day2-safety-reviewer-{SUFFIX}",
    definition=PromptAgentDefinition(model=MODEL_DEPLOYMENT, instructions=REVIEWER_INSTRUCTIONS),
)

PIPELINE_ORDER = ...  # TODO 1: the three agent names, in pipeline order.
check_todos(PIPELINE_ORDER=PIPELINE_ORDER)
print("Pipeline order:", PIPELINE_ORDER)

## 2. Build and run the sequential workflow

`SequentialBuilder` runs each participant in the order of its `participants` list. `output_from="all"` returns every stage's result, not just the last one, so you can see the draft change shape at each step.

In [ ]:
QUESTION = "Why do I need to take the full course of antibiotics even after I feel better?"

async with AsyncExitStack() as stack:
    drafter = await stack.enter_async_context(
        FoundryAgent(
            project_endpoint=PROJECT_ENDPOINT,
            agent_name=drafter_version.name,
            agent_version=str(drafter_version.version),
            credential=credential,
            name="drafter",
            allow_preview=False,
            timeout=240,
        )
    )
    simplifier = await stack.enter_async_context(
        FoundryAgent(
            project_endpoint=PROJECT_ENDPOINT,
            agent_name=simplifier_version.name,
            agent_version=str(simplifier_version.version),
            credential=credential,
            name="simplifier",
            allow_preview=False,
            timeout=240,
        )
    )
    reviewer = await stack.enter_async_context(
        FoundryAgent(
            project_endpoint=PROJECT_ENDPOINT,
            agent_name=reviewer_version.name,
            agent_version=str(reviewer_version.version),
            credential=credential,
            name="reviewer",
            allow_preview=False,
            timeout=240,
        )
    )

    workflow = SequentialBuilder(participants=[drafter, simplifier, reviewer], output_from="all").build()
    events = await asyncio.wait_for(workflow.run(QUESTION), timeout=600)

steps = events.get_outputs()
for number, step in enumerate(steps, 1):
    for message in step.messages:
        print(f"STEP {number} | {message.author_name or 'agent'}\n{message.text}\n")

## Deterministic success check

This checks the shape of the run, not the wording: three distinct agents ran, in order, and each produced a result.

In [ ]:
assert len({drafter_version.name, simplifier_version.name, reviewer_version.name}) == 3, (
    "The three agents should be distinct."
)
assert len(steps) == 3, f"Expected 3 pipeline steps, got {len(steps)}. Check output_from='all'."
assert all(step.messages for step in steps), "A pipeline step produced no message."

authors = [step.messages[-1].author_name for step in steps]
assert authors == ["drafter", "simplifier", "reviewer"], (
    f"Steps ran out of order: {authors}. Check PIPELINE_ORDER and the participants list."
)

print("PASS - three agents ran in order: draft, then simplify, then safety-review.")

## What you learned

1. `SequentialBuilder` runs participants in list order; each one sees every earlier result.
2. `output_from="all"` exposes every stage, useful while you are still checking the pipeline.
3. A longer pipeline means more model calls and more places for an error to enter. Only add a stage if it needs its own check.

**Further reading:** [Sequential orchestration](https://learn.microsoft.com/en-us/agent-framework/workflows/orchestrations/sequential) on Microsoft Learn, and [AI Agent Orchestration Patterns](https://learn.microsoft.com/en-us/azure/architecture/ai-ml/guide/ai-agent-design-patterns#sequential-orchestration) for when to choose this pattern.

**Next:** Lab 8 runs a similar set of specialist agents **concurrently** instead of in sequence.

In [ ]:
project.close()
credential.close()
print("Closed the local clients. All Foundry agents remain.")